# Retail Store Inventory: Demand Forecasting, Inventory Optimization & Dynamic Pricing

This notebook works through the three data-science challenges posed by the **Retail Store Inventory**
dataset — a synthetic yet realistic collection of **73,000+ daily records** spanning multiple stores and
products, with attributes covering sales, inventory, pricing, weather, promotions and holidays.

**Dataset features**

| Column | Description |
|---|---|
| `Date` | Daily record date |
| `Store ID` / `Product ID` | Unique store and product identifiers |
| `Category` | Product category (Electronics, Clothing, Groceries, …) |
| `Region` | Geographic region of the store |
| `Inventory Level` | Stock available at the start of the day |
| `Units Sold` | Units sold during the day |
| `Units Ordered` | Units ordered during the day |
| `Demand Forecast` | Demand predicted from past trends |
| `Price` / `Discount` / `Competitor Pricing` | Pricing attributes |
| `Weather Condition` | Daily weather impacting sales |
| `Holiday/Promotion` | Holiday / promotion indicator |
| `Seasonality` | Season label |

**Challenges addressed**

1. **Time-Series Demand Forecasting** — predict daily demand and benchmark an LSTM against classical methods (ARIMA, Prophet) and a GRU.
2. **Inventory Optimization** — derive safety stock, reorder point and Economic Order Quantity (EOQ) to balance stockouts against overstock.
3. **Dynamic Pricing** — estimate price elasticity and find a revenue-maximising price given competition and promotions.

**Notebook structure**

1. Environment setup & imports
2. Data loading & initial inspection
3. Data cleaning & preprocessing
4. Exploratory data analysis (EDA)
5. Challenge 1 — Time-series demand forecasting
6. Challenge 2 — Inventory optimization
7. Challenge 3 — Dynamic pricing
8. Integrated insights & recommendations

> **Note:** Several charts use Plotly. GitHub's static notebook viewer does not execute the embedded
> JavaScript, so interactive figures may appear blank on GitHub — open the notebook in
> Jupyter or [nbviewer](https://nbviewer.org/) to view them.

---

## 1. Environment Setup & Imports

All third-party libraries used across the notebook are imported once here. `warnings` are silenced to keep
the model-fitting output readable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, f1_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf
from prophet import Prophet
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings(action='ignore')

import plotly.express as px
import plotly.graph_objects as go 
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default='iframe'
pio.renderers.default='notebook_connected'

## 2. Data Loading & Initial Inspection

Load the raw CSV and take a first look at its shape, schema, duplicates and missing values before any
transformation.

In [ ]:
df = pd.read_csv('retail_store_inventory.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

## 3. Data Cleaning & Preprocessing

Column names are shortened to concise, code-friendly labels used throughout the rest of the notebook
(e.g. `Inventory Level` → `Inventory`, `Units Sold` → `Sales`, `Demand Forecast` → `Demand`).

In [ ]:
# renaming columns
df = df.rename(columns = {'Inventory Level':'Inventory',
                          'Units Sold':'Sales',
                          'Units Ordered':'Orders',
                          'Demand Forecast':'Demand',
                          'Weather Condition':'Weather',
                          'Holiday/Promotion':'Promotion',
                          'Competitor Pricing':'Competitor Price'})



## 4. Exploratory Data Analysis (EDA)

Box plots for every numeric column help surface the spread, central tendency and potential outliers in the
data before modelling.

In [ ]:
# box plots for numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in numerical_cols:
    fig = px.box(df, x=col)
    fig.update_layout(height=400,width=800,title_text=col)
    fig.show()

## 5. Challenge 1 — Time-Series Demand Forecasting

**Goal:** predict daily product demand across stores from historical sales and inventory data, and test
whether a deep-learning model (LSTM) can beat classical approaches.

### 5.1 Building the Aggregate Demand Series

Individual store/product series are sparse, so demand is aggregated to a single **total daily demand** series.
The `Date` column is parsed to a datetime index and the series is visualised over time.

In [ ]:
# Convert Date to datetime and set as index for time series analysis
df['Date'] = pd.to_datetime(df['Date'])
df_ts = df.set_index('Date').sort_index()

print("Date range:", df_ts.index.min(), "to", df_ts.index.max())
print("Number of unique dates:", df_ts.index.nunique())

# Create aggregate time series for total daily demand across all stores/products
daily_demand = df_ts.groupby(df_ts.index)['Demand'].sum()
print("\nAggregate daily demand shape:", daily_demand.shape)
print("First 5 days of aggregate demand:")
print(daily_demand.head())

# Plot the aggregate demand time series
fig = px.line(x=daily_demand.index, y=daily_demand.values, 
              title='Total Daily Demand Across All Stores and Products',
              labels={'x': 'Date', 'y': 'Total Demand'})
fig.show()


### 5.2 Stationarity Tests & Seasonal Decomposition

Classical models such as ARIMA assume (or require differencing to reach) stationarity. The **ADF** and
**KPSS** tests are used together — they have opposite null hypotheses, so agreement gives a more confident
verdict. An additive seasonal decomposition (weekly period) separates trend, seasonality and residual.

In [ ]:
# Check stationarity of the aggregate demand time series
print("STATIONARITY TESTS")
print("=" * 50)

# ADF Test
adf_result = adfuller(daily_demand.dropna())
print(f'ADF Statistic: {adf_result[0]:.6f}')
print(f'p-value: {adf_result[1]:.6f}')
print('Critical Values:')
for key, value in adf_result[4].items():
    print(f'   {key}: {value:.3f}')

# KPSS Test
kpss_result = kpss(daily_demand.dropna(), regression='c')
print(f'\nKPSS Statistic: {kpss_result[0]:.6f}')
print(f'p-value: {kpss_result[1]:.6f}')
print('Critical Values:')
for key, value in kpss_result[3].items():
    print(f'   {key}: {value:.3f}')

# Interpret results
print("\nINTERPRETATION:")
if adf_result[1] <= 0.05:
    print("ADF: Reject null hypothesis - series is stationary")
else:
    print("ADF: Fail to reject null hypothesis - series is non-stationary")

if kpss_result[1] >= 0.05:
    print("KPSS: Fail to reject null hypothesis - series is stationary")
else:
    print("KPSS: Reject null hypothesis - series is non-stationary")

# Seasonal decomposition
print("\nSEASONAL DECOMPOSITION")
print("=" * 50)
decomposition = seasonal_decompose(daily_demand.dropna(), model='additive', period=7)  # Weekly seasonality

# Plot decomposition
fig = make_subplots(rows=4, cols=1,
                    subplot_titles=('Original', 'Trend', 'Seasonal', 'Residual'),
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=decomposition.observed.index, y=decomposition.observed.values,
                        name='Observed', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=decomposition.trend.index, y=decomposition.trend.values,
                        name='Trend', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=decomposition.seasonal.index, y=decomposition.seasonal.values,
                        name='Seasonal', line=dict(color='orange')), row=3, col=1)
fig.add_trace(go.Scatter(x=decomposition.resid.index, y=decomposition.resid.values,
                        name='Residual', line=dict(color='red')), row=4, col=1)

fig.update_layout(height=800, title_text="Time Series Decomposition of Aggregate Demand")
fig.show()


### 5.3 ACF / PACF Analysis

Autocorrelation (ACF) and partial autocorrelation (PACF) plots guide the choice of the AR (`p`) and
MA (`q`) orders for ARIMA.

In [ ]:
# Calculate ACF and PACF for ARIMA parameter identification
plt.figure(figsize=(14, 6))

# ACF
plt.subplot(1, 2, 1)
plot_acf(daily_demand.dropna(), lags=50, ax=plt.gca())
plt.title('Autocorrelation Function (ACF)')
plt.xlabel('Lag')
plt.ylabel('ACF')

# PACF
plt.subplot(1, 2, 2)
plot_pacf(daily_demand.dropna(), lags=50, ax=plt.gca())
plt.title('Partial Autocorrelation Function (PACF)')
plt.xlabel('Lag')
plt.ylabel('PACF')

plt.tight_layout()
plt.show()

# Suggest ARIMA parameters based on plots
print("ARIMA PARAMETER SUGGESTIONS (based on ACF/PACF plots):")
print("Look for significant spikes in ACF and PACF to determine p, q values")
print("Consider differencing if series is non-stationary")


### 5.4 ARIMA Model

A small grid search over `(p, d, q)` selects the order with the lowest **AIC**. The fitted model forecasts
the held-out test window and is scored with MAE, RMSE and MAPE. The last 10% of the series is reserved for testing.

In [ ]:
# Prepare data for ARIMA - ensure no missing values
ts_data = daily_demand.dropna()
print(f"Time series length: {len(ts_data)}")

# Split data into train and test (last 30 days for testing)
train_size = int(len(ts_data) * 0.9)
train_data = ts_data[:train_size]
test_data = ts_data[train_size:]

print(f"Training data: {len(train_data)} points ({train_data.index.min()} to {train_data.index.max()})")
print(f"Testing data: {len(test_data)} points ({test_data.index.min()} to {test_data.index.max()})")

# Try different ARIMA orders and select best based on AIC
best_aic = float('inf')
best_order = None
best_model = None

# Grid search for p, d, q values (limited range for efficiency)
p_values = range(0, 4)
d_values = range(0, 3)
q_values = range(0, 4)

print("\nGRID SEARCH FOR BEST ARIMA PARAMETERS")
print("=" * 40)

for p in p_values:
    for d in d_values:
        for q in q_values:
            try:
                if p == 0 and d == 0 and q == 0:
                    continue  # Skip trivial model
                model = ARIMA(train_data, order=(p, d, q))
                fitted_model = model.fit()
                if fitted_model.aic < best_aic:
                    best_aic = fitted_model.aic
                    best_order = (p, d, q)
                    best_model = fitted_model
                # print(f"ARIMA({p},{d},{q}) AIC: {fitted_model.aic:.2f}")
            except:
                continue

print(f"Best ARIMA{best_order} AIC: {best_aic:.2f}")

# Forecast using the best model
forecast_steps = len(test_data)
forecast = best_model.forecast(steps=forecast_steps)
forecast_index = test_data.index

# Calculate evaluation metrics
mae = mean_absolute_error(test_data, forecast)
rmse = np.sqrt(mean_squared_error(test_data, forecast))
mape = np.mean(np.abs((test_data - forecast) / test_data)) * 100

print(f"\nARIMA{best_order} FORECAST EVALUATION")
print("=" * 40)
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.4f}%")

# Plot actual vs forecast
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_data.index, y=train_data.values,
                        name='Training Data', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test_data.index, y=test_data.values,
                        name='Actual', line=dict(color='green')))
fig.add_trace(go.Scatter(x=forecast_index, y=forecast.values,
                        name='ARIMA Forecast', line=dict(color='red', dash='dash')))

fig.update_layout(title=f'ARIMA{best_order} Forecast vs Actual Demand',
                  xaxis_title='Date',
                  yaxis_title='Demand',
                  hovermode='x unified')
fig.show()


### 5.5 Prophet Model

Facebook Prophet is fitted with weekly and yearly seasonality. MAPE is computed with a zero-guard so that
days with zero demand do not blow up the metric.

In [ ]:
# Prepare data for Prophet (requires 'ds' and 'y' columns)
prophet_df = pd.DataFrame({
    'ds': daily_demand.index,
    'y': daily_demand.values
})

# Split data for Prophet
train_prophet = prophet_df.iloc[:train_size]
test_prophet = prophet_df.iloc[train_size:]

print(f"Prophet training data: {len(train_prophet)} points")
print(f"Prophet testing data: {len(test_prophet)} points")

# Initialize and fit Prophet model
# Adding weekly and yearly seasonality, daily if appropriate
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive'
)

# Add custom seasonalities if needed
# prophet_model.add_seasonality(name='monthly', period=30.5, fourier_order=5)

print("\nFitting Prophet model...")
prophet_model.fit(train_prophet)

# Create future dataframe for forecasting
future = prophet_model.make_future_dataframe(periods=len(test_prophet))
forecast = prophet_model.predict(future)

# Extract forecast for test period
prophet_forecast = forecast.iloc[train_size:][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
prophet_forecast.set_index('ds', inplace=True)

# Calculate evaluation metrics with protection against division by zero
mae_prophet = mean_absolute_error(test_prophet['y'], prophet_forecast['yhat'])
rmse_prophet = np.sqrt(mean_squared_error(test_prophet['y'], prophet_forecast['yhat']))

# Safe MAPE calculation: avoid division by zero
y_true = test_prophet['y'].values
y_pred = prophet_forecast['yhat'].values

# Only calculate MAPE for non-zero actual values
non_zero_mask = y_true != 0
if non_zero_mask.any():
    mape_prophet = np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
    zero_count = (~non_zero_mask).sum()
    if zero_count > 0:
        print(f"\nWarning: {zero_count} zero actual values excluded from MAPE calculation")
else:
    mape_prophet = np.nan
    print("\nWarning: All actual values are zero - MAPE cannot be calculated")

print(f"\nPROPHET MODEL EVALUATION")
print("=" * 40)
print(f"MAE:  {mae_prophet:.4f}")
print(f"RMSE: {rmse_prophet:.4f}")
if not np.isnan(mape_prophet):
    print(f"MAPE: {mape_prophet:.4f}%")
else:
    print("MAPE: Cannot be calculated (zero actual values)")

# Plot Prophet forecast with improved readability
fig = prophet_model.plot(forecast)
fig.set_size_inches(14, 8)  # Larger figure size
plt.title('Prophet Forecast of Aggregate Demand', fontsize=16, pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot components with improved readability
fig2 = prophet_model.plot_components(forecast)
fig2.set_size_inches(14, 10)  # Larger figure size for components
plt.suptitle('Prophet Model Components', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


### 5.6 LSTM Model

The series is framed as a supervised problem with 30-day input sequences. To avoid **data leakage**, the
`MinMaxScaler` is fitted on the training portion only. The network uses stacked LSTM layers with dropout
and L2 regularisation, a Huber loss (robust to outliers), early stopping and learning-rate reduction.

In [ ]:
# Prepare data for LSTM - need to create sequences
def create_sequences(data, seq_length):
    """Create sequences for LSTM input"""
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Scale the data for LSTM - fit scaler ONLY on training data to prevent leakage
scaler_lstm = MinMaxScaler(feature_range=(0, 1))

# Split data FIRST, then scale to avoid data leakage
train_values = ts_data[:train_size].values.reshape(-1, 1)
test_values = ts_data[train_size:].values.reshape(-1, 1)

# Fit scaler on training data only
scaler_lstm.fit(train_values)
train_scaled = scaler_lstm.transform(train_values).flatten()
test_scaled = scaler_lstm.transform(test_values).flatten()

# Combine scaled data for sequence creation (but remember the boundary)
full_scaled = np.concatenate([train_scaled, test_scaled])

# Choose sequence length (e.g., 30 days of history to predict next day)
SEQ_LENGTH = 30
X, y = create_sequences(full_scaled, SEQ_LENGTH)

print(f"LSTM data shape - X: {X.shape}, y: {y.shape}")

# Now split X and y into train/test maintaining the chronological boundary
# The last training sequence ends at index: train_size - SEQ_LENGTH
train_end_idx = train_size - SEQ_LENGTH
if train_end_idx <= 0:
    raise ValueError(f"Training size ({train_size}) must be greater than SEQ_LENGTH ({SEQ_LENGTH})")

X_train, X_test = X[:train_end_idx], X[train_end_idx:]
y_train, y_test = y[:train_end_idx], y[train_end_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Reshape for LSTM [samples, time steps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Build LSTM model with regularization
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LENGTH, 1),
         kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.3),
    LSTM(32, return_sequences=False,
         kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='huber')  # Huber loss is more robust to outliers

# Define callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=0.00001, verbose=1)

print("\nTraining LSTM model...")
history = lstm_model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=200,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Make predictions
train_predict = lstm_model.predict(X_train, verbose=0)
test_predict = lstm_model.predict(X_test, verbose=0)

# Inverse transform predictions
train_predict = scaler_lstm.inverse_transform(train_predict)
y_train_inv = scaler_lstm.inverse_transform(y_train.reshape(-1, 1))
test_predict = scaler_lstm.inverse_transform(test_predict)
y_test_inv = scaler_lstm.inverse_transform(y_test.reshape(-1, 1))

# Calculate metrics (using test set)
lstm_mae = mean_absolute_error(y_test_inv, test_predict)
lstm_rmse = np.sqrt(mean_squared_error(y_test_inv, test_predict))
# Safe MAPE calculation
non_zero_mask = y_test_inv.flatten() != 0
if non_zero_mask.any():
    lstm_mape = np.mean(np.abs((y_test_inv[non_zero_mask] - test_predict[non_zero_mask]) / y_test_inv[non_zero_mask])) * 100
else:
    lstm_mape = np.nan

print(f"\nLSTM MODEL EVALUATION")
print("=" * 40)
print(f"MAE:  {lstm_mae:.4f}")
print(f"RMSE: {lstm_rmse:.4f}")
if not np.isnan(lstm_mape):
    print(f"MAPE: {lstm_mape:.4f}%")
else:
    print("MAPE: Cannot be calculated (zero actual values)")

# Plot training history with better visualization
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('LSTM Model Loss', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Plot predictions vs actual with CORRECT INDEX ALIGNMENT
plt.subplot(1, 2, 2)

# Get CORRECT dates for test predictions
# The first test prediction corresponds to index: train_size
test_start_date = ts_data.index[train_size]
test_dates = ts_data.index[train_size:train_size + len(test_predict)]

plt.plot(test_dates, y_test_inv, label='Actual', color='blue', linewidth=2)
plt.plot(test_dates, test_predict, label='LSTM Forecast', color='red', linestyle='--', linewidth=2)
plt.title('LSTM Forecast vs Actual (Test Period)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Additional diagnostic: plot residuals
residuals = y_test_inv.flatten() - test_predict.flatten()
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(test_dates, residuals, color='purple', alpha=0.7)
plt.title('LSTM Forecast Residuals', fontsize=12)
plt.xlabel('Date', fontsize=10)
plt.ylabel('Residual', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
plt.title('Residual Distribution', fontsize=12)
plt.xlabel('Residual Value', fontsize=10)
plt.ylabel('Frequency', fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show some prediction examples
print(f"\nPREDICTION EXAMPLES (first 5 test points):")
print("Date\t\tActual\t\tPredicted\tError")
print("-" * 50)
for i in range(min(5, len(test_dates))):
    date_str = test_dates[i].strftime('%Y-%m-%d')
    actual = y_test_inv[i, 0]
    pred = test_predict[i, 0]
    error = actual - pred
    print(f"{date_str}\t{actual:.2f}\t\t{pred:.2f}\t\t{error:.2f}")


### 5.7 GRU Model

A GRU with a comparable architecture is trained on the same sequences, providing a second deep-learning
baseline against the LSTM.

In [ ]:
# Build GRU model (similar architecture to LSTM but with GRU layers)
gru_model = Sequential([
    GRU(50, return_sequences=True, input_shape=(SEQ_LENGTH, 1)),
    Dropout(0.2),
    GRU(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

gru_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print("\nTraining GRU model...")
history_gru = gru_model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Make predictions with GRU
gru_train_predict = gru_model.predict(X_train)
gru_test_predict = gru_model.predict(X_test)

# Inverse transform predictions
gru_train_predict = scaler_lstm.inverse_transform(gru_train_predict)
gru_test_predict = scaler_lstm.inverse_transform(gru_test_predict)

# Calculate metrics
gru_mae = mean_absolute_error(y_test_inv, gru_test_predict)
gru_rmse = np.sqrt(mean_squared_error(y_test_inv, gru_test_predict))
gru_mape = np.mean(np.abs((y_test_inv - gru_test_predict) / y_test_inv)) * 100

print(f"\nGRU MODEL EVALUATION")
print("=" * 40)
print(f"MAE:  {gru_mae:.4f}")
print(f"RMSE: {gru_rmse:.4f}")
print(f"MAPE: {gru_mape:.4f}%")

# Plot GRU training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_gru.history['loss'], label='Training Loss')
plt.plot(history_gru.history['val_loss'], label='Validation Loss')
plt.title('GRU Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot GRU predictions vs actual
plt.subplot(1, 2, 2)
plt.plot(test_dates, y_test_inv, label='Actual', color='blue')
plt.plot(test_dates, gru_test_predict, label='GRU Forecast', color='red', linestyle='--')
plt.title('GRU Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.legend()
plt.tight_layout()
plt.show()


### 5.8 Model Comparison

MAE, RMSE and MAPE for all four models (ARIMA, Prophet, LSTM, GRU) are collected into one table and the
best model is selected by lowest MAPE.

In [ ]:
# Create a comparison table of all models
models_comparison = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'LSTM', 'GRU'],
    'MAE': [mae, mae_prophet, lstm_mae, gru_mae],
    'RMSE': [rmse, rmse_prophet, lstm_rmse, gru_rmse],
    'MAPE (%)': [mape, mape_prophet, lstm_mape, gru_mape]
})

print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(models_comparison.round(4))

# Visualize comparison
fig = go.Figure()

models = models_comparison['Model']
mae_values = models_comparison['MAE']
rmse_values = models_comparison['RMSE']

fig.add_trace(go.Bar(name='MAE', x=models, y=mae_values, marker_color='lightblue'))
fig.add_trace(go.Bar(name='RMSE', x=models, y=rmse_values, marker_color='lightcoral'))

fig.update_layout(
    title='Model Comparison: MAE and RMSE',
    xaxis_title='Model',
    yaxis_title='Error Value',
    barmode='group',
    hovermode='x unified'
)
fig.show()

# Find best model based on MAPE
best_model_idx = models_comparison['MAPE (%)'].idxmin()
best_model = models_comparison.iloc[best_model_idx]

print(f"\nBEST PERFORMING MODEL: {best_model['Model']}")
print(f"MAPE: {best_model['MAPE (%)']:.4f}%")


## 6. Challenge 2 — Inventory Optimization

**Goal:** set inventory policies that minimise stockouts while avoiding overstock. Using demand statistics
and standard supply-chain formulas, we compute the **safety stock**, **reorder point** and
**Economic Order Quantity (EOQ)**, then break down the resulting annual ordering and holding costs.
The cost, lead-time and service-level parameters below are illustrative assumptions and can be replaced
with real business figures.

In [ ]:
print("CHALLENGE 2: INVENTORY OPTIMIZATION")
print("=" * 50)

# For inventory optimization, we need:
# 1. Demand forecasts (from Challenge 1)
# 2. Lead time assumptions
# 3. Holding costs and stockout costs
# 4. Service level targets

# Use the best forecasting model's predictions for inventory calculations
# Let's use the Prophet model forecasts as an example (you can change this)

# Get the forecasts from our best model (Prophet in this case)
forecast_demand = prophet_forecast['yhat'].values
forecast_dates = prophet_forecast.index

# Historical data for calculating variability
historical_demand = ts_data.values

print(f"Historical demand statistics:")
print(f"  Mean: {np.mean(historical_demand):.2f}")
print(f"  Std:  {np.std(historical_demand):.2f}")
print(f"  Min:  {np.min(historical_demand):.2f}")
print(f"  Max:  {np.max(historical_demand):.2f}")

print(f"\nForecast demand statistics (next {len(forecast_demand)} days):")
print(f"  Mean: {np.mean(forecast_demand):.2f}")
print(f"  Std:  {np.std(forecast_demand):.2f}")


In [ ]:
# Inventory parameters (these would typically come from business data)
LEAD_TIME_DAYS = 7  # Supplier lead time in days
HOLDING_COST_RATE = 0.25  # 25% of item value per year
STOCKOUT_COST = 50.0  # Cost per unit of stockout (lost sales, penalty, etc.)
ITEM_COST = 25.0  # Average cost per item ($)
SERVICE_LEVEL = 0.95  # Target service level (95%)

# Convert annual holding cost to daily
holding_cost_per_day = ITEM_COST * HOLDING_COST_RATE / 365

# Calculate demand during lead time
# For simplicity, we'll use average daily demand * lead time
avg_daily_demand = np.mean(historical_demand)
demand_during_lt = avg_daily_demand * LEAD_TIME_DAYS

# Calculate safety stock using service level approach
# Safety stock = Z * σ_demand * sqrt(LT)
# Where Z is the z-score for desired service level
from scipy import stats
z_score = stats.norm.ppf(SERVICE_LEVEL)  # Z-score for service level
demand_std = np.std(historical_demand)
safety_stock = z_score * demand_std * np.sqrt(LEAD_TIME_DAYS)

# Reorder point = demand during lead time + safety stock
reorder_point = demand_during_lt + safety_stock

# Economic Order Quantity (EOQ)
# EOQ = sqrt((2 * D * S) / H)
# Where D = annual demand, S = ordering cost, H = holding cost per unit per year
ANNUAL_DEMAND = avg_daily_demand * 365
ORDERING_COST = 50.0  # Cost per order (admin, shipping, etc.)
holding_cost_per_year = ITEM_COST * HOLDING_COST_RATE

eoq = np.sqrt((2 * ANNUAL_DEMAND * ORDERING_COST) / holding_cost_per_year)

print("INVENTORY OPTIMIZATION RESULTS")
print("=" * 40)
print(f"Lead Time: {LEAD_TIME_DAYS} days")
print(f"Average Daily Demand: {avg_daily_demand:.2f} units")
print(f"Demand Std Dev: {demand_std:.2f} units")
print(f"Service Level Target: {SERVICE_LEVEL*100:.0f}%")
print(f"Z-Score: {z_score:.3f}")
print(f"Safety Stock: {safety_stock:.2f} units")
print(f"Demand During Lead Time: {demand_during_lt:.2f} units")
print(f"Reorder Point: {reorder_point:.2f} units")
print(f"Economic Order Quantity (EOQ): {eoq:.2f} units")
print(f"Holding Cost/Day: ${holding_cost_per_day:.4f}")
print(f"Stockout Cost/Unit: ${STOCKOUT_COST:.2f}")

# Calculate total annual cost
annual_ordering_cost = (ANNUAL_DEMAND / eoq) * ORDERING_COST
annual_holding_cost = (eoq / 2) * holding_cost_per_year
annual_total_cost = annual_ordering_cost + annual_holding_cost

print(f"\nANNUAL COST BREAKDOWN (EOQ Policy)")
print("=" * 40)
print(f"Annual Ordering Cost: ${annual_ordering_cost:.2f}")
print(f"Annual Holding Cost: ${annual_holding_cost:.2f}")
print(f"Total Annual Cost: ${annual_total_cost:.2f}")


## 7. Challenge 3 — Dynamic Pricing

**Goal:** build a pricing strategy that maximises revenue given demand, competitor pricing and discounts.
Price elasticity is estimated with a log-log regression, promotion lift is measured, and a simple
revenue = price × demand(price) model is optimised to find the revenue-maximising price.

In [ ]:
print("CHALLENGE 3: DYNAMIC PRICING")
print("=" * 40)

# For dynamic pricing, we analyze:
# 1. Price elasticity of demand
# 2. Relationship between price, promotions, and demand
# 3. Competitor pricing effects
# 4. Optimal pricing strategies

# Prepare data for pricing analysis
pricing_df = df.copy()

# Ensure we have the necessary columns
required_cols = ['Price', 'Demand', 'Discount', 'Competitor Price', 'Promotion']
missing_cols = [col for col in required_cols if col not in pricing_df.columns]
if missing_cols:
    print(f"Warning: Missing columns for pricing analysis: {missing_cols}")
else:
    print("All required columns present for pricing analysis")

# Basic statistics
print("PRICE AND DEMAND STATISTICS")
print("=" * 30)
print(f"Price - Mean: ${pricing_df['Price'].mean():.2f}, Std: ${pricing_df['Price'].std():.2f}")
print(f"Demand - Mean: {pricing_df['Demand'].mean():.2f} units, Std: {pricing_df['Demand'].std():.2f}")
print(f"Discount - Mean: {pricing_df['Discount'].mean():.2f}%")
print(f"Competitor Price - Mean: ${pricing_df['Competitor Price'].mean():.2f}")

# Price elasticity analysis (log-log regression)
# log(Demand) = α + β * log(Price) + ε
# Where β is the price elasticity

# Filter out zero prices or demands for log transformation
pricing_clean = pricing_df[(pricing_df['Price'] > 0) & (pricing_df['Demand'] > 0)].copy()
pricing_clean['log_price'] = np.log(pricing_clean['Price'])
pricing_clean['log_demand'] = np.log(pricing_clean['Demand'])

# Simple linear regression for elasticity
from sklearn.linear_model import LinearRegression

X_elasticity = pricing_clean['log_price'].values.reshape(-1, 1)
y_elasticity = pricing_clean['log_demand'].values

elasticity_model = LinearRegression()
elasticity_model.fit(X_elasticity, y_elasticity)

price_elasticity = elasticity_model.coef_[0]
r_squared = elasticity_model.score(X_elasticity, y_elasticity)

print(f"\nPRICE ELASTICITY ANALYSIS")
print("=" * 30)
print(f"Price Elasticity: {price_elasticity:.4f}")
print(f"R-squared: {r_squared:.4f}")
print(f"Interpretation: A 1% increase in price leads to {abs(price_elasticity):.2f}% change in demand")

# If elasticity is negative (normal case), demand decreases with price increase
if price_elasticity < 0:
    print("Normal goods: Demand decreases as price increases")
else:
    print("Warning: Positive elasticity suggests Giffen/Veblen goods or model issues")

# Visualize price vs demand relationship
fig = px.scatter(pricing_clean, x='Price', y='Demand', 
                 trendline='ols',
                 title='Price vs Demand Relationship',
                 labels={'Price': 'Price ($)', 'Demand': 'Demand (units)'})
fig.show()

# Analyze promotion effectiveness
promo_effect = pricing_df.groupby('Promotion')['Demand'].mean()
print(f"\nPROMOTION EFFECTIVENESS")
print("=" * 30)
print(f"Average demand without promotion: {promo_effect.get(0, 0):.2f} units")
print(f"Average demand with promotion: {promo_effect.get(1, 0):.2f} units")
if 1 in promo_effect.index and 0 in promo_effect.index:
    promo_lift = ((promo_effect[1] - promo_effect[0]) / promo_effect[0]) * 100
    print(f"Promotional lift: {promo_lift:.2f}%")


In [ ]:
# Simple price optimization model
# Objective: Maximize Revenue = Price * Demand(Price)
# Constraint: Price must be within reasonable bounds

def demand_function(price, base_demand=None, elasticity=None, competitor_price=None, promo_effect=0):
    """
    Simple demand function incorporating price, competition, and promotions
    """
    if base_demand is None:
        base_demand = np.mean(pricing_df['Demand'])
    if elasticity is None:
        elasticity = price_elasticity  # From our earlier calculation
    if competitor_price is None:
        competitor_price = np.mean(pricing_df['Competitor Price'])
    
    # Base demand adjusted by price elasticity
    price_ratio = price / competitor_price if competitor_price > 0 else 1
    demand = base_demand * (price_ratio ** elasticity)
    
    # Promotion effect (simplified)
    demand *= (1 + promo_effect)
    
    return max(0, demand)  # Demand can't be negative

def revenue_function(price, base_demand=None, elasticity=None, competitor_price=None):
    """Calculate revenue for a given price"""
    demand = demand_function(price, base_demand, elasticity, competitor_price)
    return price * demand

# Find optimal price using simple optimization
price_range = np.linspace(pricing_df['Price'].min() * 0.5, 
                         pricing_df['Price'].max() * 1.5, 1000)

revenues = [revenue_function(p) for p in price_range]
optimal_price_idx = np.argmax(revenues)
optimal_price = price_range[optimal_price_idx]
max_revenue = revenues[optimal_price_idx]

# Current average price and revenue
current_price = np.mean(pricing_df['Price'])
current_demand = demand_function(current_price)
current_revenue = current_price * current_demand

print("PRICE OPTIMIZATION RESULTS")
print("=" * 30)
print(f"Current Average Price: ${current_price:.2f}")
print(f"Current Estimated Demand: {current_demand:.2f} units")
print(f"Current Revenue: ${current_revenue:.2f}")
print(f"")
print(f"Optimal Price: ${optimal_price:.2f}")
print(f"Demand at Optimal Price: {demand_function(optimal_price):.2f} units")
print(f"Maximum Revenue: ${max_revenue:.2f}")
print(f"")
print(f"Price Change: {((optimal_price - current_price) / current_price) * 100:.2f}%")
print(f"Revenue Improvement: {((max_revenue - current_revenue) / current_revenue) * 100:.2f}%")

# Visualize revenue vs price
fig = go.Figure()
fig.add_trace(go.Scatter(x=price_range, y=revenues, 
                        mode='lines', name='Revenue'))
fig.add_trace(go.Scatter(x=[current_price], y=[current_revenue],
                        mode='markers', marker=dict(size=10, color='red'),
                        name='Current Price'))
fig.add_trace(go.Scatter(x=[optimal_price], y=[max_revenue],
                        mode='markers', marker=dict(size=10, color='green'),
                        name='Optimal Price'))

fig.update_layout(
    title='Revenue vs Price Optimization',
    xaxis_title='Price ($)',
    yaxis_title='Revenue ($)',
    hovermode='x unified'
)
fig.show()

# Consider competitor pricing in optimization
print(f"\nCompetitor Analysis:")
print(f"Average Competitor Price: ${np.mean(pricing_df['Competitor Price']):.2f}")
print(f"Our Optimal Price vs Competitor: {((optimal_price - np.mean(pricing_df['Competitor Price'])) / np.mean(pricing_df['Competitor Price'])) * 100:.2f}%")


## 8. Integrated Insights & Recommendations

Findings from all three challenges are combined into a single summary dashboard and a set of actionable
recommendations.

In [ ]:
print("INTEGRATED INSIGHTS AND RECOMMENDATIONS")
print("=" * 50)

# Create a summary dashboard
print("KEY FINDINGS FROM ALL THREE CHALLENGES:")
print()

print("1. DEMAND FORECASTING (Challenge 1)")
print("-" * 40)
print(f"   Best Model: {best_model['Model']}")
print(f"   Forecast Accuracy (MAPE): {best_model['MAPE (%)']:.2f}%")
print(f"   This means our forecasts are typically off by ±{best_model['MAPE (%)']:.2f}%")
print()

print("2. INVENTORY OPTIMIZATION (Challenge 2)")
print("-" * 40)
print(f"   Recommended Safety Stock: {safety_stock:.0f} units")
print(f"   Recommended Reorder Point: {reorder_point:.0f} units")
print(f"   Economic Order Quantity: {eoq:.0f} units per order")
print(f"   This minimizes holding + ordering costs")
print()

print("3. DYNAMIC PRICING (Challenge 3)")
print("-" * 40)
print(f"   Price Elasticity: {price_elasticity:.3f}")
print(f"   Optimal Price: ${optimal_price:.2f} (vs current ${current_price:.2f})")
print(f"   Potential Revenue Increase: {((max_revenue - current_revenue) / current_revenue) * 100:.1f}%")
print()

print("INTEGRATED RECOMMENDATIONS:")
print("-" * 30)
print("1. Use the best forecasting model for demand predictions")
print("2. Set inventory policies based on forecast variability and service levels")
print("3. Adjust prices dynamically based on demand elasticity and competitor actions")
print("4. Monitor forecast accuracy and update models regularly")
print("5. Consider promotional timing based on demand patterns")

# Create a combined visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Forecast Accuracy Comparison', 
                   'Inventory Cost Components',
                   'Price Elasticity Relationship',
                   'Revenue Optimization'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "scatter"}]]
)

# Model comparison bar chart
fig.add_trace(
    go.Bar(x=models_comparison['Model'], y=models_comparison['MAE'], 
           name='MAE', marker_color='lightblue'),
    row=1, col=1
)

# Inventory costs
inventory_costs = ['Ordering Cost', 'Holding Cost', 'Total Cost']
inventory_values = [annual_ordering_cost, annual_holding_cost, annual_total_cost]
fig.add_trace(
    go.Bar(x=inventory_costs, y=inventory_values,
           name='Costs ($)', marker_color='lightcoral'),
    row=1, col=2
)

# Price elasticity scatter
fig.add_trace(
    go.Scatter(x=pricing_clean['Price'], y=pricing_clean['Demand'],
               mode='markers', name='Price-Demand Data',
               marker=dict(color='green', size=4)),
    row=2, col=1
)
# Add trend line
x_range = np.linspace(pricing_clean['Price'].min(), pricing_clean['Price'].max(), 100)
y_trend = np.exp(elasticity_model.intercept_ + elasticity_model.coef_[0] * np.log(x_range))
fig.add_trace(
    go.Scatter(x=x_range, y=y_trend, mode='lines',
               name=f'Elasticity = {price_elasticity:.3f}',
               line=dict(color='red', dash='dash')),
    row=2, col=1
)

# Revenue optimization
fig.add_trace(
    go.Scatter(x=price_range, y=revenues,
               mode='lines', name='Revenue vs Price',
               line=dict(color='purple')),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=[optimal_price], y=[max_revenue],
               mode='markers', marker=dict(size=12, color='gold'),
               name='Optimal Point'),
    row=2, col=2
)

fig.update_layout(height=800, showlegend=True, 
                  title_text="Integrated Analytics Dashboard")
fig.show()


---

## Conclusion

This notebook delivers an end-to-end analytics workflow for retail inventory:

- **Forecasting** — four models (ARIMA, Prophet, LSTM, GRU) are trained and compared on a common test window, with leakage-safe scaling and robust error metrics.
- **Inventory** — forecast variability feeds classical safety-stock, reorder-point and EOQ calculations to balance service level against cost.
- **Pricing** — price elasticity and a revenue-optimisation curve suggest a data-driven price relative to competitors and promotions.

**Possible next steps:** per-store / per-category forecasting instead of a single aggregate series,
hyperparameter tuning for the neural models, exogenous regressors (weather, holidays, promotions) in
Prophet/ARIMAX, and back-testing the inventory and pricing policies on held-out periods.